# 01 · LLM eval metrics: exact match, F1, BLEU, ROUGE, edit distance, semantic similarity, perplexity

After this notebook you can implement every classic text-generation metric from scratch, say in one sentence what each one really measures, and show with numbers where each one lies. That lets you pick the right metric for a task instead of reaching for BLEU by reflex.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min (uses two small cached Hugging Face models) · **Needs:** [Embeddings and similarity](../07_genai_foundations/02_embeddings_and_similarity.ipynb)

## Why this matters in interviews

- "How would you evaluate this feature?" comes up in nearly every GenAI loop, and the first follow-up is "why not just use BLEU or ROUGE?". You need to say what they measure and show where they break.
- Token-level F1, ROUGE-L and edit distance are classic "code it now" exercises. Each one is a `Counter` or a small dynamic-programming table.
- Picking the metric per task (exact match for IDs, WER for speech, pass@k for code, a judge for open-ended answers) is a senior signal.
- Perplexity and logprobs come up whenever pretraining, fine-tuning or confidence-based routing is discussed.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ev04` | Compare the main evaluation methods: exact match, semantic similarity, LLM judge, pairwise, RAGAS, human, and online A/B. |
| `ev01` | What are the fundamentals of AI evaluation? Walk me through the whole picture. |
| `ev16` | How do you evaluate a summarisation feature? |
| `ev17` | How do you evaluate a classification task done by an LLM? |
| `ev18` | How do you evaluate when there is no single right answer? |
| `ev21` | How would you evaluate a code-generating feature? |
| `fo23` | What are logprobs and what can you legitimately use them for? |
| `ve31` | What is the difference between semantic similarity and relevance? |
| `tn20` | How do you evaluate a fine-tuned model before shipping it? |

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import functools
import itertools
import math
import re
import string
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
rng = np.random.default_rng(0)

## The map: what each metric family actually measures

| Family | Metrics | Needs a reference? | What it really measures |
|---|---|---|---|
| Surface overlap | exact match, token F1, BLEU, ROUGE, chrF | yes | shared strings and n-grams |
| Character edits | Levenshtein distance, CER, WER | yes | how many keystrokes apart two strings are |
| Meaning | embedding cosine, BERTScore | yes | topical and semantic closeness |
| Likelihood | perplexity, logprobs | no | how unsurprised a language model is by the text |
| Behaviour | task assertions, pass@k (run the code) | a spec or tests | whether the output does the job |
| Judgement | LLM judge, human rating | optional | whatever the rubric says ([next notebook](02_llm_as_judge_and_its_biases.ipynb)) |

**In plain English:** the first four rows compare *strings*. Only the last two ask whether the answer is *right*. Keep that split in mind for the rest of the notebook.

## 1. Normalise before you compare

Exact match on raw strings punishes capital letters, full stops and "the". The SQuAD benchmark fixed this with a four-step normaliser: lowercase, strip punctuation, drop the articles *a/an/the*, squash whitespace. Almost every QA metric uses it, and a different normaliser can move exact-match scores by several points. So always say which one you used.

In [ ]:
PUNCT = set(string.punctuation)
ARTICLES = re.compile(r"\b(a|an|the)\b")

def normalize(text: str) -> str:
    """SQuAD-style answer normalisation: lowercase, drop punctuation, drop articles, squash spaces."""
    text = text.lower()
    text = "".join(ch for ch in text if ch not in PUNCT)
    text = ARTICLES.sub(" ", text)
    return " ".join(text.split())

for s in ["The Eiffel Tower!", "  eiffel   TOWER ", "Eiffel-Tower", "5 days", "five days"]:
    print(f"{s!r:22} -> {normalize(s)!r}")

assert normalize("The Eiffel Tower!") == normalize("  eiffel   TOWER ") == "eiffel tower"
assert normalize("Eiffel-Tower") == "eiffeltower"      # gotcha: deleting punctuation glues words together
assert normalize("5 days") != normalize("five days")     # normalisation knows nothing about numbers

## 2. Exact match and token-level F1 (SQuAD style)

- **Exact match (EM):** 1 if the normalised prediction equals a normalised reference, else 0. It is strict, cheap and unambiguous. It is right for IDs, labels and short factual answers.
- **Token F1:** treat both answers as bags of words. Precision = shared tokens / prediction tokens, recall = shared tokens / reference tokens, F1 = their harmonic mean. It gives partial credit when the model wraps the right answer in extra words.
- With several acceptable references, take the **max** over references.

Check one by hand before trusting the code. `"Paris is the capital of France"` normalises to `[paris, is, capital, of, france]` (5 tokens), the reference to `[paris]`. One token is shared, so precision = 1/5, recall = 1/1, and F1 = 2·(0.2·1)/(0.2+1) = **1/3**. The asserts check that and three more hand-computed values.

In [ ]:
def exact_match(pred: str, refs) -> float:
    """1.0 if the normalised prediction equals any normalised reference."""
    refs = [refs] if isinstance(refs, str) else refs
    return float(any(normalize(pred) == normalize(r) for r in refs))

def token_f1(pred: str, refs) -> float:
    """SQuAD token-level F1 on normalised bags of words (clipped counts); best over references."""
    refs = [refs] if isinstance(refs, str) else refs
    def f1(p, r):
        p, r = normalize(p).split(), normalize(r).split()
        if not p or not r:                      # "no answer" cases: correct only if both are empty
            return float(p == r)
        same = sum((Counter(p) & Counter(r)).values())
        if same == 0:
            return 0.0
        precision, recall = same / len(p), same / len(r)
        return 2 * precision * recall / (precision + recall)
    return max(f1(pred, r) for r in refs)

for pred, ref in [("The Eiffel Tower.", "Eiffel Tower"), ("It is the Eiffel Tower in Paris", "Eiffel Tower"),
                  ("Paris is the capital of France", "Paris"), ("London", "Paris")]:
    print(f"EM={exact_match(pred, ref):.0f}  F1={token_f1(pred, ref):.3f}   pred={pred!r:34} ref={ref!r}")

assert abs(token_f1("Paris is the capital of France", "Paris") - 1 / 3) < 1e-12
# [it, is, eiffel, tower, in, paris] vs [eiffel, tower]: P = 2/6, R = 2/2 -> F1 = 0.5
assert abs(token_f1("It is the Eiffel Tower in Paris", "Eiffel Tower") - 0.5) < 1e-12
assert exact_match("The Eiffel Tower.", "Eiffel Tower") == 1.0 and exact_match("It is the Eiffel Tower in Paris", "Eiffel Tower") == 0.0
assert abs(token_f1("London", ["Paris", "London, UK"]) - 2 / 3) < 1e-12   # best reference wins
assert token_f1("", "") == 1.0 and token_f1("", "Paris") == 0.0

## 3. BLEU from scratch

**In plain English:** BLEU counts how many of the candidate's word n-grams (n = 1..4) also appear in a reference. It penalises candidates that are too short.

The mechanism (Papineni et al., 2002):
1. **Clipped (modified) n-gram precision** p_n: each candidate n-gram counts at most as many times as it appears in a single reference. Without clipping, `the the the the` would get perfect unigram precision.
2. **Geometric mean** of p_1..p_4, so one zero precision makes the whole score zero.
3. **Brevity penalty** BP = 1 if the candidate is longer than the reference, else exp(1 − r/c). Precision alone would reward saying almost nothing.

BLEU = BP · exp( (1/4) Σ log p_n ). Papers report it × 100. Real reported BLEU uses **sacreBLEU** with its standard tokenizer, so the numbers are comparable. Our tokenizer here is a simple lowercase word split. The cell after the implementation runs three worked examples, with the hand calculation in the comments.

In [ ]:
def tokens(text: str) -> list[str]:
    """Lowercase word tokens with punctuation dropped (a simple stand-in for sacreBLEU's '13a' tokenizer)."""
    return re.findall(r"[a-z0-9]+", text.lower())

def ngram_counts(toks, n):
    return Counter(tuple(toks[i:i + n]) for i in range(len(toks) - n + 1))

def clipped_matches(cand, refs, n):
    """BLEU's modified precision numerator and denominator for n-grams of size n."""
    c = ngram_counts(cand, n)
    max_ref = Counter()
    for r in refs:
        max_ref |= ngram_counts(r, n)                     # element-wise max over references
    return sum(min(cnt, max_ref[g]) for g, cnt in c.items()), sum(c.values())

def brevity_penalty(c_len, r_len):
    return 1.0 if c_len > r_len else (math.exp(1 - r_len / c_len) if c_len else 0.0)

def closest_ref_len(c_len, refs):
    return min((abs(len(r) - c_len), len(r)) for r in refs)[1]

def bleu(candidate: str, references, max_n=4, smooth=False) -> dict:
    refs = [references] if isinstance(references, str) else references
    cand, refs = tokens(candidate), [tokens(r) for r in refs]
    precisions = []
    for n in range(1, max_n + 1):
        match, total = clipped_matches(cand, refs, n)
        if total == 0:                  # candidate shorter than n: use only the orders it has ("effective order")
            break
        if smooth and n > 1:            # add-one smoothing for n >= 2 (Lin & Och, 2004)
            match, total = match + 1, total + 1
        precisions.append(match / total)
    bp = brevity_penalty(len(cand), closest_ref_len(len(cand), refs))
    if not precisions or min(precisions) == 0:
        return {"bleu": 0.0, "precisions": precisions, "bp": bp}
    return {"bleu": bp * math.exp(sum(map(math.log, precisions)) / len(precisions)), "precisions": precisions, "bp": bp}

In [ ]:
# (a) Papineni's degenerate candidate: 'the' occurs 7 times but only twice in the reference -> p1 = 2/7
res = bleu("the the the the the the the", "the cat is on the mat", max_n=1)
assert abs(res["precisions"][0] - 2 / 7) < 1e-12
print(f"(a) clipped unigram precision = {res['precisions'][0]:.3f}   (unclipped it would be 7/7 = 1.0)")

# (b) One inflection changed. Hand count: p1 = 8/9, p2 = 6/8, p3 = 4/7, p4 = 2/6, BP = 1 (same length)
REF = "the quick brown fox jumps over the lazy dog"
res = bleu("the quick brown fox jumped over the lazy dog", REF)
assert all(abs(a - b) < 1e-12 for a, b in zip(res["precisions"], [8 / 9, 6 / 8, 4 / 7, 2 / 6]))
assert res["bp"] == 1.0 and abs(res["bleu"] - (8 / 9 * 6 / 8 * 4 / 7 * 2 / 6) ** 0.25) < 1e-12
print(f"(b) jumps -> jumped: p_n = {[round(p, 3) for p in res['precisions']]}, BLEU = {res['bleu']:.3f}")

# (c) Truncation: every n-gram matches (p_n = 1) but the brevity penalty is exp(1 - 9/4)
res = bleu("the quick brown fox", REF)
assert res["precisions"] == [1.0, 1.0, 1.0, 1.0] and abs(res["bleu"] - math.exp(1 - 9 / 4)) < 1e-12
print(f"(c) first 4 words only: precisions all 1.0, BP = {res['bp']:.3f}, BLEU = {res['bleu']:.3f}")

In [ ]:
variants = {"identical": REF, "one word changed": "the quick brown fox jumped over the lazy dog",
            "truncated (first 4 words)": "the quick brown fox", "word order shuffled": "the lazy dog jumps over the quick brown fox"}
scores = {name: bleu(text, REF) for name, text in variants.items()}
x = np.arange(4)
fig, ax = plt.subplots(figsize=(11, 4))
for i, (name, r) in enumerate(scores.items()):
    ax.bar(x + i * 0.2 - 0.3, r["precisions"], width=0.2, label=f"{name} (BLEU {r['bleu']:.2f})")
ax.set_xticks(x, [f"p{n} ({n}-grams)" for n in range(1, 5)])
ax.set_ylabel("clipped precision"); ax.set_ylim(0, 1.1)
ax.set_title("BLEU's n-gram precisions: small edits hurt the long n-grams most")
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1)); plt.tight_layout(); plt.show()

**Read the chart:** the one-word change keeps 8 of 9 unigrams, but every 3-gram and 4-gram that touches "jumped" breaks. So BLEU falls to about 0.6. A shuffled sentence keeps all its unigrams and loses most 4-grams. BLEU is a *surface* metric, and word order only counts through the higher n-grams.

### Sentence BLEU vs corpus BLEU

BLEU was designed as a **corpus** metric. You sum the clipped matches and totals over all sentences, then combine them once. At sentence level, a single missing 4-gram gives exactly 0. That is why sentence-level BLEU needs smoothing, and why the mean of sentence BLEUs is not corpus BLEU.

In [ ]:
def corpus_bleu(cands, refs_list, max_n=4):
    match, total, c_len, r_len = [0] * max_n, [0] * max_n, 0, 0
    for cand, refs in zip(cands, refs_list):
        ct, rt = tokens(cand), [tokens(r) for r in ([refs] if isinstance(refs, str) else refs)]
        for n in range(1, max_n + 1):
            m, t = clipped_matches(ct, rt, n)
            match[n - 1] += m; total[n - 1] += t
        c_len += len(ct); r_len += closest_ref_len(len(ct), rt)
    precisions = [m / t for m, t in zip(match, total)]
    return 0.0 if min(precisions) == 0 else brevity_penalty(c_len, r_len) * math.exp(sum(map(math.log, precisions)) / max_n)

cands = ["the cat sat on the mat", "the quick brown fox jumped over the lazy dog", "refunds take five business days"]
refs = ["the cat is on the mat", REF, "refunds take five business days"]
sent = [bleu(c, r)["bleu"] for c, r in zip(cands, refs)]
sent_smooth = [bleu(c, r, smooth=True)["bleu"] for c, r in zip(cands, refs)]
print("sentence BLEU (unsmoothed):", [round(s, 3) for s in sent], " mean =", round(float(np.mean(sent)), 3))
print("sentence BLEU (add-one)   :", [round(s, 3) for s in sent_smooth], " mean =", round(float(np.mean(sent_smooth)), 3))
print("corpus BLEU               :", round(corpus_bleu(cands, refs), 3))
assert sent[0] == 0.0 and sent_smooth[0] > 0          # 'the cat sat on the mat' shares no 4-gram with the reference
assert corpus_bleu(cands, refs) > np.mean(sent)       # pooling counts is not averaging scores

## 4. ROUGE-1, ROUGE-2, ROUGE-L from scratch

**In plain English:** ROUGE is BLEU's recall-oriented cousin. It was built for summarisation and asks how much of the *reference* shows up in the candidate.

- **ROUGE-N:** overlap of n-grams. Recall = overlap / reference n-grams, precision = overlap / candidate n-grams. Most libraries report the F1.
- **ROUGE-L:** uses the **longest common subsequence** (LCS). That is the longest run of words appearing in both texts *in the same order*, but not necessarily next to each other. It rewards order without demanding exact n-grams.
- The `rouge-score` package also offers stemming and **ROUGE-Lsum**, which splits summaries on newlines. The core arithmetic is the same as below.

In [ ]:
def prf(overlap, n_cand, n_ref):
    p = overlap / n_cand if n_cand else 0.0
    r = overlap / n_ref if n_ref else 0.0
    return {"p": p, "r": r, "f": 0.0 if overlap == 0 else 2 * p * r / (p + r)}

def rouge_n(candidate, reference, n=1):
    c, r = ngram_counts(tokens(candidate), n), ngram_counts(tokens(reference), n)
    return prf(sum((c & r).values()), sum(c.values()), sum(r.values()))

def lcs_len(a, b):
    """Longest common subsequence length by dynamic programming, O(len(a) * len(b))."""
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i, x in enumerate(a, 1):
        for j, y in enumerate(b, 1):
            dp[i][j] = dp[i - 1][j - 1] + 1 if x == y else max(dp[i - 1][j], dp[i][j - 1])
    return dp[-1][-1]

def rouge_l(candidate, reference):
    c, r = tokens(candidate), tokens(reference)
    return prf(lcs_len(c, r), len(c), len(r))

ref, cand, shuffled = "the cat is on the mat", "the cat sat on the mat", "mat the on is cat the"
for name, text in [("one word changed", cand), ("same words, shuffled", shuffled)]:
    r1, r2, rl = rouge_n(text, ref, 1)["f"], rouge_n(text, ref, 2)["f"], rouge_l(text, ref)["f"]
    print(f"{name:22} ROUGE-1 F={r1:.3f}  ROUGE-2 F={r2:.3f}  ROUGE-L F={rl:.3f}")

# Hand checks. 'the cat sat on the mat' vs 'the cat is on the mat':
#   unigram overlap 5 of 6 each side -> 5/6; bigram overlap {the cat, on the, the mat} = 3 of 5 -> 0.6;
#   LCS = 'the cat on the mat' = 5 words -> 5/6
assert abs(rouge_n(cand, ref, 1)["f"] - 5 / 6) < 1e-12
assert abs(rouge_n(cand, ref, 2)["f"] - 3 / 5) < 1e-12
assert abs(rouge_l(cand, ref)["f"] - 5 / 6) < 1e-12
# Shuffling keeps every unigram (ROUGE-1 = 1.0), but the LCS drops to 3 words (e.g. 'the on the') -> 0.5
assert rouge_n(shuffled, ref, 1)["f"] == 1.0 and abs(rouge_l(shuffled, ref)["f"] - 0.5) < 1e-12

## 5. Edit distance, character error rate (CER) and word error rate (WER)

**Levenshtein distance** is the minimum number of single-item insertions, deletions and substitutions that turn one sequence into another. It is a textbook dynamic program: cell (i, j) holds the distance between the first i items of the source and the first j of the target.

- **CER** = character edits / reference length. It is the standard metric for OCR and for extracted strings.
- **WER** = word edits / reference word count. It is the standard for speech recognition. It **can exceed 1.0** when the hypothesis has many insertions.

Watch the invoice-ID line in the output. A CER of 0.15 *sounds* like a small error, but the ID is simply wrong. For identifiers, codes, amounts and dates, use **exact match per field**.

In [ ]:
def levenshtein(a, b) -> int:
    """Minimum insertions + deletions + substitutions turning a into b (strings or token lists)."""
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i]
        for j, y in enumerate(b, 1):
            cur.append(min(prev[j] + 1,               # delete x
                           cur[j - 1] + 1,            # insert y
                           prev[j - 1] + (x != y)))   # substitute (free when equal)
        prev = cur
    return prev[-1]

def cer(pred, ref):
    return levenshtein(pred, ref) / max(len(ref), 1)

def wer(pred, ref):
    return levenshtein(pred.split(), ref.split()) / max(len(ref.split()), 1)

print(f"kitten -> sitting: {levenshtein('kitten', 'sitting')} edits")
print(f"ASR  'please cancel my border' vs 'please cancel my order': WER={wer('please cancel my border', 'please cancel my order'):.2f}  "
      f"CER={cer('please cancel my border', 'please cancel my order'):.3f}")
print(f"ID   'INV-2024-0024' vs 'INV-2024-0042': CER={cer('INV-2024-0024', 'INV-2024-0042'):.3f}  exact match={exact_match('INV-2024-0024', 'INV-2024-0042'):.0f}")
print(f"WER of 'a b c d e' vs reference 'a' = {wer('a b c d e', 'a'):.1f}  (insertions push WER above 1)")

assert levenshtein("kitten", "sitting") == 3 and levenshtein("", "abc") == 3 and levenshtein(["a", "b"], ["a", "b"]) == 0
assert abs(wer("please cancel my border", "please cancel my order") - 1 / 4) < 1e-12
assert levenshtein("INV-2024-0024", "INV-2024-0042") == 2          # swapped digits = 2 substitutions
assert wer("a b c d e", "a") == 4.0

In [ ]:
def lev_matrix(a, b):
    D = np.zeros((len(a) + 1, len(b) + 1), dtype=int)
    D[:, 0], D[0, :] = np.arange(len(a) + 1), np.arange(len(b) + 1)
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            D[i, j] = min(D[i - 1, j] + 1, D[i, j - 1] + 1, D[i - 1, j - 1] + (a[i - 1] != b[j - 1]))
    return D

src, tgt = "kitten", "sitting"
D = lev_matrix(src, tgt)
assert D[-1, -1] == levenshtein(src, tgt) == 3
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.imshow(D, cmap="Blues")
for i in range(D.shape[0]):
    for j in range(D.shape[1]):
        ax.text(j, i, D[i, j], ha="center", va="center", color="white" if D[i, j] > 4 else "black")
ax.set_xticks(range(len(tgt) + 1), ["''"] + list(tgt)); ax.set_yticks(range(len(src) + 1), ["''"] + list(src))
ax.set_xlabel(f"target: {tgt}"); ax.set_ylabel(f"source: {src}")
ax.set_title("Levenshtein DP table: each cell = cheapest edit of the prefixes; bottom-right = 3"); plt.show()

## 6. Semantic similarity with a sentence encoder

**In plain English:** embed the reference and the candidate with a sentence encoder, then take the cosine of the two vectors. Paraphrases land close together even when they share no words. This is exactly what the lexical metrics above cannot do.

We use `sentence-transformers/all-MiniLM-L6-v2` (384-dim, 90 MB, loaded from the local cache). The usual one-liner is `SentenceTransformer(name).encode(texts, normalize_embeddings=True)`. Here we do its three steps with plain `transformers` instead: tokenise, run the encoder, mean-pool over real tokens and L2-normalise. That gives the same vectors (to about 1e-8), keeps the import light, and hands us the per-token vectors that **BERTScore** needs.

**BERTScore** (Zhang et al., 2020) matches each token of one text to its most similar token in the other. Precision averages over candidate tokens, recall over reference tokens. The real metric uses a larger model, optional idf weights and a rescaling baseline. Below is the same idea on MiniLM.

In [ ]:
import torch

def load_encoder(name="sentence-transformers/all-MiniLM-L6-v2"):
    """MiniLM tokenizer + encoder from the local Hugging Face cache; downloads ~90 MB once if missing."""
    try:
        from transformers import AutoModel, AutoTokenizer
        from transformers.utils import logging as hf_logging
        hf_logging.disable_progress_bar()
        try:
            return AutoTokenizer.from_pretrained(name, local_files_only=True), AutoModel.from_pretrained(name, local_files_only=True).eval()
        except Exception:
            return AutoTokenizer.from_pretrained(name), AutoModel.from_pretrained(name).eval()
    except Exception as e:
        print(f"Could not load MiniLM ({type(e).__name__}: {e}); falling back to TF-IDF character n-grams.")
        return None, None

enc_tok, encoder = load_encoder()

@functools.lru_cache(maxsize=None)
def token_vectors(text):
    """Encoder output for one text: all hidden states, and the real tokens (no [CLS]/[SEP]) L2-normalised."""
    batch = enc_tok(text, return_tensors="pt")
    with torch.no_grad():
        h = encoder(**batch).last_hidden_state[0]
    return h, torch.nn.functional.normalize(h[1:-1], dim=1)

@functools.lru_cache(maxsize=None)
def sentence_vector(text):
    if encoder is None:                                  # fallback: character n-gram TF-IDF (fitted on this text)
        return None
    h, _ = token_vectors(text)
    return torch.nn.functional.normalize(h.mean(0), dim=0).numpy()   # mean-pool (single text: no padding), normalise

def cosine(a: str, b: str) -> float:
    if encoder is None:
        from sklearn.feature_extraction.text import TfidfVectorizer
        X = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit_transform([a, b]).toarray()
        return float(X[0] @ X[1] / (np.linalg.norm(X[0]) * np.linalg.norm(X[1])))
    return float(sentence_vector(a) @ sentence_vector(b))

def bertscore_like(cand: str, ref: str) -> float:
    """Greedy token matching of contextual embeddings: the BERTScore idea, on MiniLM."""
    if encoder is None:
        return float("nan")
    sim = (token_vectors(cand)[1] @ token_vectors(ref)[1].T).numpy()
    p, r = sim.max(axis=1).mean(), sim.max(axis=0).mean()
    return float(2 * p * r / (p + r))

print("encoder:", "all-MiniLM-L6-v2 via transformers" if encoder is not None else "TF-IDF fallback")

In [ ]:
REFERENCE = "Refunds are processed within 5 business days."
probes = {"correct paraphrase": "You'll get your money back in five working days.",
          "wrong number": "Refunds are processed within 50 business days.",
          "negated": "Refunds are not processed within 5 business days.",
          "unrelated": "Our store opens at 9 am on weekdays."}
sims = {k: cosine(v, REFERENCE) for k, v in probes.items()}
for k, v in probes.items():
    print(f"{k:18}  cosine={sims[k]:.3f}  BERTScore-style={bertscore_like(v, REFERENCE):.3f}   {v!r}")

if encoder is not None:     # the claim below is about MiniLM, not the fallback
    assert sims["wrong number"] > sims["correct paraphrase"] and sims["negated"] > sims["correct paraphrase"]
    assert sims["unrelated"] < sims["correct paraphrase"]

This is the `ev04` trap in numbers. The encoder knows that "money back in five working days" is about refunds. But it scores the **wrong** answers ("50 days", "not processed") *higher* than the correct paraphrase, because they share nearly every token. Embedding similarity measures topic, not truth. It is blind to numbers and negation. Never use it alone for factual correctness (`ve31`: similarity is not relevance, and it is not correctness either).

## 7. Perplexity: how surprised is a language model?

**In plain English:** perplexity is "how many tokens the model was effectively choosing between" at each step. It is the exponent of the average negative log-probability:

PPL = exp( −(1/N) Σ log p(token_i | previous tokens) )

A model guessing uniformly over V tokens has PPL = V. A perfect model has PPL = 1. Perplexity needs **no reference answer**. It is the standard way to track pretraining and fine-tuning on held-out text.

After the two hand checks, we use a real (tiny) language model, `HuggingFaceTB/SmolLM2-135M-Instruct`, on CPU. We compute per-token log-probs by hand from the logits (the logits at position t predict token t+1), then check the result against the `loss` Hugging Face computes when you pass `labels`.

In [ ]:
def perplexity_from_logprobs(logprobs) -> float:
    """exp of the mean negative log-probability per token."""
    return math.exp(-sum(logprobs) / len(logprobs))

probs = [0.5, 0.25, 0.125, 1.0]            # product = 1/64 over 4 tokens -> PPL = 64 ** (1/4)
ppl = perplexity_from_logprobs([math.log(p) for p in probs])
print(f"token probs {probs} -> perplexity {ppl:.4f} (= 64^(1/4) = {64 ** 0.25:.4f})")
assert abs(ppl - 64 ** 0.25) < 1e-12
V = 50
assert abs(perplexity_from_logprobs([math.log(1 / V)] * 10) - V) < 1e-9     # uniform guessing over V -> PPL = V
print(f"uniform guessing over {V} tokens -> perplexity {V}")

In [ ]:
torch.manual_seed(0)

def load_lm(name="HuggingFaceTB/SmolLM2-135M-Instruct"):
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        try:
            tok = AutoTokenizer.from_pretrained(name, local_files_only=True)
            lm = AutoModelForCausalLM.from_pretrained(name, local_files_only=True, dtype=torch.float32)
        except Exception:
            tok = AutoTokenizer.from_pretrained(name)
            lm = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)   # checkpoint is bf16
        return tok, lm.eval()
    except Exception as e:
        print(f"Could not load {name} ({type(e).__name__}: {e}); skipping the tiny-model demo.")
        return None, None

lm_tok, lm = load_lm()

def token_logprobs(text):
    ids = lm_tok(text, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = lm(ids).logits
    logp = torch.log_softmax(logits[0, :-1], dim=-1)          # prediction for token t+1 is made at position t
    target = ids[0, 1:]
    return lm_tok.convert_ids_to_tokens(target), logp[torch.arange(len(target)), target].numpy(), ids

sentences = {"fluent, true": "The capital of France is Paris.",
             "fluent, false": "The capital of France is Berlin.",
             "same words, shuffled": "Paris France capital the is of.",
             "support answer": "Refunds are processed within 5 business days."}
ppls, per_token = {}, {}
if lm is not None:
    for name, s in sentences.items():
        toks, lps, ids = token_logprobs(s)
        ppls[name], per_token[name] = perplexity_from_logprobs(lps), (toks, lps)
        print(f"{name:22} PPL = {ppls[name]:9.1f}   ({len(toks)} predicted tokens)  {s!r}")
    with torch.no_grad():                                         # our maths == the library's loss
        hf_ppl = float(torch.exp(lm(ids, labels=ids).loss))
    assert math.isclose(ppls[name], hf_ppl, rel_tol=1e-4)
    assert ppls["fluent, true"] < ppls["fluent, false"] < ppls["same words, shuffled"]

In [ ]:
if lm is not None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
    for ax, name in zip(axes, ["fluent, true", "fluent, false"]):
        toks, lps = per_token[name]
        ax.bar(range(len(toks)), -np.array(lps) / math.log(2), color="#3b5bdb")
        ax.set_xticks(range(len(toks)), [t.replace("Ġ", "") for t in toks], rotation=30)
        ax.set_title(f"{name}: PPL {ppls[name]:.0f}"); ax.set_xlabel("predicted token")
    axes[0].set_ylabel("surprisal, bits = -log2 p")
    fig.suptitle("Per-token surprisal: the false fact costs the model extra bits at one token")
    plt.tight_layout(); plt.show()

What perplexity does and does not tell you:

- The shuffled sentence is thousands of times more surprising. Perplexity is an excellent **fluency / in-distribution** signal.
- The false sentence is *more* surprising than the true one, but still fluent-looking. Its perplexity is nowhere near the shuffled one. A fluent falsehood can have low perplexity, so **perplexity is not a correctness metric** for answers.
- Perplexity depends on the tokenizer. Models with different vocabularies are not comparable. Use bits-per-byte or bits-per-character across tokenizers.

### Logprobs from an API

The same number comes out of an API when it exposes logprobs. `gpt-4.1-mini` does. Many reasoning models do not, so don't build a system around a signal your final model may not give you (`fo23`). A legitimate use is flagging low-confidence spans or routing to a stronger model. An illegitimate one is treating mean logprob as a hallucination detector: confident fabrications are exactly where it fails.

In [ ]:
resp = client.chat.completions.create(
    model=MODEL, temperature=0, max_tokens=40, logprobs=True, top_logprobs=2,
    messages=[{"role": "user", "content": "In one short sentence: how long do refunds usually take?"}])
content = resp.choices[0].logprobs.content
lps = [t.logprob for t in content]
print("answer        :", resp.choices[0].message.content)
print(f"tokens scored : {len(lps)}   mean logprob: {np.mean(lps):.3f}   answer perplexity: {perplexity_from_logprobs(lps):.2f}")
print("least likely  :", [(t.token, round(t.logprob, 2)) for t in sorted(content, key=lambda t: t.logprob)[:3]])
if PROVIDER == "offline":
    print("(offline stand-in: the text and logprobs are placeholders, but the maths and the API shape are real)")

## 8. The showdown: 8 answers, one reference, every metric

One question, one reference answer, eight candidates. A human has marked each candidate right or wrong. Before running the next cell, **predict** which metrics will score "wrong number" above "paraphrase".

We add one more column: a **key-fact check**. It is a five-line task-specific assertion: the answer must contain the reference's numbers (with number words mapped to digits) and must not add a negation. It is deliberately tailored to this one question. That is what assertions at the base of the eval pyramid look like (`ev01`).

After the heatmap, one number per metric: how well does it **separate** right answers from wrong ones? ROC AUC over the 8 candidates answers exactly that. 1.0 means every correct answer outscores every wrong one; 0.5 is a coin flip. Eight items is a toy sample, but the pattern is the one you see at scale.

In [ ]:
QUESTION = "How long does a refund take?"
CANDIDATES = [  # (label, answer, does a human say it is correct?)
    ("copy of reference", "Refunds are processed within 5 business days.", True),
    ("paraphrase", "You'll get your money back in five working days.", True),
    ("terse", "5 business days.", True),
    ("verbose but right", "Once we receive the returned item, refunds are usually processed within 5 business days.", True),
    ("wrong number", "Refunds are processed within 50 business days.", False),
    ("negated", "Refunds are not processed within 5 business days.", False),
    ("on topic, no answer", "Refunds are issued to the original payment method.", False),
    ("off topic", "Our store opens at 9 am on weekdays.", False),
]
WORD_NUMS = {"one": "1", "two": "2", "three": "3", "four": "4", "five": "5", "six": "6", "seven": "7", "ten": "10"}
NEGATIONS = {"not", "no", "never", "cannot", "isn't", "aren't", "won't", "don't"}

def key_fact_check(answer: str, reference: str) -> float:
    """Task-specific assertion: same numbers as the reference, and no negation the reference lacks."""
    def nums(t):
        return {WORD_NUMS.get(w, w) for w in tokens(t) if w.isdigit() or w in WORD_NUMS}
    def negated(t):
        return any(w in NEGATIONS for w in re.findall(r"[a-z']+", t.lower()))
    return float(nums(answer) == nums(reference) and negated(answer) == negated(reference))

METRICS = {
    "exact match": lambda a: exact_match(a, REFERENCE),
    "token F1": lambda a: token_f1(a, REFERENCE),
    "BLEU-4 (smoothed)": lambda a: bleu(a, REFERENCE, smooth=True)["bleu"],
    "ROUGE-1 F": lambda a: rouge_n(a, REFERENCE, 1)["f"],
    "ROUGE-L F": lambda a: rouge_l(a, REFERENCE)["f"],
    "1 - CER": lambda a: max(0.0, 1 - cer(normalize(a), normalize(REFERENCE))),
    "MiniLM cosine": lambda a: cosine(a, REFERENCE),
    "BERTScore-style": lambda a: bertscore_like(a, REFERENCE),
    "key-fact check": lambda a: key_fact_check(a, REFERENCE),
}
table = pd.DataFrame([{"candidate": label, "human says": "correct" if ok else "wrong", **{m: f(ans) for m, f in METRICS.items()}}
                      for label, ans, ok in CANDIDATES]).set_index("candidate")
table

In [ ]:
M = table[list(METRICS)].to_numpy(dtype=float)
fig, ax = plt.subplots(figsize=(11, 4.5))
im = ax.imshow(M, cmap="viridis", vmin=0, vmax=1, aspect="auto")
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        ax.text(j, i, f"{M[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if M[i, j] < 0.55 else "black")
ax.set_xticks(range(len(METRICS)), list(METRICS), rotation=25, ha="right")
ax.set_yticks(range(len(CANDIDATES)), [f"{'CORRECT' if ok else 'WRONG'} · {label}" for label, _, ok in CANDIDATES])
ax.axhline(3.5, color="red", lw=2)
plt.colorbar(im, ax=ax, label="score")
ax.set_title("Above the red line: correct answers. Below: wrong. Lexical metrics can't tell them apart.")
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score

human = np.array([ok for *_, ok in CANDIDATES], dtype=int)
auc = pd.Series({m: roc_auc_score(human, table[m]) for m in METRICS if table[m].notna().all()}).sort_values()
print(auc.round(3).to_string())

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.barh(auc.index, auc.values, color=["#2f9e44" if v == 1 else "#868e96" for v in auc.values])
ax.axvline(0.5, ls="--", color="black", lw=1); ax.text(0.51, -0.6, "coin flip", fontsize=8)
ax.set_xlim(0, 1.05); ax.set_xlabel("ROC AUC: correct vs wrong answers")
ax.set_title("Only the task-specific assertion separates right from wrong here"); plt.show()

lexical = ["token F1", "BLEU-4 (smoothed)", "ROUGE-1 F", "ROUGE-L F", "1 - CER"] + (["MiniLM cosine"] if encoder is not None else [])
for m in lexical:
    assert table.loc["wrong number", m] > table.loc["paraphrase", m], m     # wrong-but-similar beats right-but-different
assert auc["key-fact check"] == 1.0 and (auc.drop("key-fact check") < 1.0).all()

**What the table says:**

- **False negatives:** the correct paraphrase gets EM 0 and low F1/BLEU/ROUGE, because it shares almost no words with the reference.
- **False positives:** "50 business days" and "not processed" score near the top on every lexical metric *and* on cosine. One character or one word flips the meaning and barely moves the score.
- **Verbosity:** the verbose-but-right answer loses precision-based points (BLEU, F1) for being helpful.
- **The assertion wins, but only here.** The key-fact check is perfect on this question and useless on the next one. For open-ended answers you need a reference-guided judge ([next notebook](02_llm_as_judge_and_its_biases.ipynb)), and for grounded answers a faithfulness check ([notebook 03](03_rag_evaluation_retrieval_and_faithfulness.ipynb)).

## 9. Which metric for which task

| Task | Primary metric | Why | Watch out for |
|---|---|---|---|
| Short factual / extractive QA | EM + token F1 (SQuAD normaliser) | the answer is a span; F1 gives partial credit | paraphrases and number words score 0 |
| IDs, codes, amounts, JSON fields | exact match **per field**, after normalisation | one wrong character = wrong answer | CER looks small on wrong IDs |
| Classification / routing by an LLM | accuracy, per-class P/R/F1, confusion matrix (`ev17`) | it *is* a classifier | labels outside the allowed set (format failures) |
| Machine translation | BLEU / chrF via sacreBLEU, COMET | standard and comparable across papers | only comparable with the same tokenizer |
| Summarisation | faithfulness + coverage (judge or rubric); ROUGE as a legacy number (`ev16`) | ROUGE rewards copying the reference | ROUGE says nothing about hallucination |
| Speech / OCR | WER / CER | edit distance is the natural error model | WER can exceed 100% |
| Code generation | pass@k by running tests in a sandbox (`ev21`) | behaviour is what matters | BLEU on code is close to meaningless |
| Open-ended chat / writing | pairwise LLM judge + a human sample (`ev18`) | no single reference exists | judge biases ([notebook 02](02_llm_as_judge_and_its_biases.ipynb)) |
| RAG answers | faithfulness, answer relevance, correctness | split retrieval from generation | [notebook 03](03_rag_evaluation_retrieval_and_faithfulness.ipynb) |
| LM / fine-tune quality | perplexity on held-out text (`tn20`) | cheap and reference-free | not comparable across tokenizers; not truth |
| "Same meaning?" | embedding cosine / BERTScore (`ve31`) | tolerant of paraphrase | blind to numbers and negation |

## 10. pass@k for code

For code, run the tests. If you generate n samples per problem and c of them pass, the unbiased estimate of "at least one of k samples passes" (Chen et al., 2021, the HumanEval paper) is

pass@k = 1 − C(n−c, k) / C(n, k)

The naive 1 − (1 − c/n)^k is biased when k is close to n. For a product, **pass@1 at temperature 0** is the number users feel. pass@10 is a research number.

In [ ]:
from math import comb

def pass_at_k(n: int, c: int, k: int) -> float:
    """Unbiased pass@k: probability that a random k-subset of the n samples contains a passing one."""
    return 1.0 if n - c < k else 1.0 - comb(n - c, k) / comb(n, k)

n, c = 10, 3
samples = [1] * c + [0] * (n - c)
for k in range(1, 6):             # brute force: enumerate every k-subset of the 10 samples
    subsets = list(itertools.combinations(samples, k))
    assert abs(pass_at_k(n, c, k) - sum(any(s) for s in subsets) / len(subsets)) < 1e-12
print("pass@k matches brute-force enumeration for k = 1..5")

ks = np.arange(1, 11)
fig, ax = plt.subplots(figsize=(7, 3.6))
for c in (1, 3, 5):
    ax.plot(ks, [pass_at_k(10, c, k) for k in ks], marker="o", label=f"{c} of 10 samples pass")
ax.set_xlabel("k (samples the user may try)"); ax.set_ylabel("pass@k")
ax.set_title("pass@k rises fast with k: quote pass@1 for a product"); ax.legend(); plt.show()

## Try it yourself

**1. chrF.** chrF (Popović, 2015) is a character n-gram F-score with β = 2, which weights recall twice as much as precision. It is popular for translation because it is robust to inflection. Implement chrF with character n-grams of size 1–6 (spaces removed) and score the eight candidates. Does "paraphrase" do better than under BLEU?

In [ ]:
# Solution — try it yourself first, then run this
def chrf(cand: str, ref: str, max_n=6, beta=2.0) -> float:
    c, r = cand.lower().replace(" ", ""), ref.lower().replace(" ", "")
    ps, rs = [], []
    for n in range(1, max_n + 1):
        cn, rn = Counter(c[i:i + n] for i in range(len(c) - n + 1)), Counter(r[i:i + n] for i in range(len(r) - n + 1))
        if not cn or not rn:
            continue
        overlap = sum((cn & rn).values())
        ps.append(overlap / sum(cn.values())); rs.append(overlap / sum(rn.values()))
    p, rec = np.mean(ps), np.mean(rs)
    return 0.0 if p + rec == 0 else float((1 + beta ** 2) * p * rec / (beta ** 2 * p + rec))

for label, ans, ok in CANDIDATES:
    print(f"{'correct' if ok else 'wrong  '}  {label:20} chrF={chrf(ans, REFERENCE):.3f}   BLEU={table.loc[label, 'BLEU-4 (smoothed)']:.3f}")
# chrF is kinder to inflections and word forms, so the paraphrase gains a little -
# but the two wrong answers still score above 0.9. Character overlap is still overlap.

**2. Multiple references.** Add a second reference, "You will be refunded in five working days.", and recompute BLEU and token F1 for "paraphrase" and "wrong number". Does a second reference rescue the paraphrase? Does it fix the false positive?

In [ ]:
# Solution — try it yourself first, then run this
refs2 = [REFERENCE, "You will be refunded in five working days."]
answers = {label: ans for label, ans, _ in CANDIDATES}
for label in ["paraphrase", "wrong number"]:
    ans = answers[label]
    print(f"{label:13} BLEU: 1 ref {bleu(ans, REFERENCE, smooth=True)['bleu']:.3f} -> 2 refs {bleu(ans, refs2, smooth=True)['bleu']:.3f}   "
          f"F1: 1 ref {token_f1(ans, REFERENCE):.3f} -> 2 refs {token_f1(ans, refs2):.3f}")
# More references reduce false negatives (paraphrases), but do nothing about false positives:
# '50 business days' still matches the first reference almost perfectly.

**3. Break the assertion.** Find an answer that is wrong but passes `key_fact_check`. Then write down what that tells you about hand-written assertions.

In [ ]:
# Solution — try it yourself first, then run this
tricky = ["Refunds are rejected within 5 business days.",
          "Shipping takes 5 business days.",
          "Refunds take 5 business days, or sometimes 5 weeks."]
for t in tricky:
    print(f"key_fact_check={key_fact_check(t, REFERENCE):.0f}   {t!r}")
# All three pass, and all three are wrong or misleading. Assertions are precise about what they check
# and blind to everything else: combine them with a judge, and grow them from real failures.

## Say it in an interview

**30-second answer (`ev04`):** "Lexical metrics (exact match, F1, BLEU, ROUGE) measure string overlap with a reference. They're cheap, deterministic and fine as regression signals when the answer space is narrow. But they miss correct paraphrases and reward wrong-but-similar answers: a changed number or a negation barely moves them. Embedding similarity fixes paraphrase but is still blind to numbers and negation. So for correctness I use task-specific assertions plus a reference-guided LLM judge calibrated on human labels. For code I run the tests. For grounded answers I check faithfulness against the context. Perplexity I use for model fluency, never for answer truth."

**Formulas to have ready:**
- SQuAD F1 = 2PR/(P+R) on normalised bags of words. EM after lowercase / punctuation / articles / whitespace.
- BLEU = BP · exp(¼ Σ log p_n), with clipped n-gram precision. BP = exp(1 − r/c) when c ≤ r.
- ROUGE-N = n-gram overlap (recall-oriented, usually reported as F). ROUGE-L uses the longest common subsequence.
- CER/WER = edits / reference length (can exceed 1). PPL = exp(−mean log p). Uniform over V gives V.
- pass@k = 1 − C(n−c, k)/C(n, k).

**Follow-ups they ask:**
- *Why clipping in BLEU?* So repeating a common word cannot inflate precision.
- *Why a brevity penalty instead of recall?* With several references, recall is ill-defined. BP stops the "say one safe word" exploit.
- *Why is sentence BLEU unreliable?* One missing 4-gram zeroes it. It was designed for corpora.
- *Can you compare perplexity across models?* Only with the same tokenizer; otherwise use bits per byte.
- *When is BLEU fine?* Translation, reported with sacreBLEU and the same references, tracked over time.

**Traps:**
- Reporting EM without naming the normaliser.
- Using cosine similarity as a correctness score (`ve31`: similarity is not relevance or truth).
- Quoting ROUGE as evidence a summary is faithful (`ev16`).
- Quoting pass@10 for a feature where the user sees one answer (`ev21`).

## Next

- [02 · LLM-as-judge and its biases](02_llm_as_judge_and_its_biases.ipynb): grading what these metrics cannot.
- [03 · RAG evaluation: retrieval and faithfulness](03_rag_evaluation_retrieval_and_faithfulness.ipynb)
- [05 · A/B testing and statistics for evals](05_ab_testing_and_statistics_for_evals.ipynb): is a 2-point gain real?
- [Metrics, confusion matrix, ROC/PR](../05_machine_learning/04_metrics_confusion_matrix_roc_pr.ipynb) for LLM-as-classifier tasks
- [Sentence-transformers embeddings and cross-encoders](../10_huggingface/04_sentence_transformers_embeddings_and_cross_encoders.ipynb)
- [Eval harness from scratch](../17_interview_coding_rounds/05_eval_harness_from_scratch.ipynb) (interview coding round)
- Theory: [genai_flow course](../../genai_flow/index.html) · [interview bank](../../ai_interview_prep/index.html)